# Exp06 · riwayat sensor sebelum episode
Run all pada CPU untuk kedua target. Dataset harus berisi paket v3, v4 dan folder v2_session_phase/window_15s_step_5s/ dengan good_acceptable.csv dan all_data.csv.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
DATASET_ROOT = PROJECT_ROOT / 'Dataset'
OUTPUT_ROOT = PROJECT_ROOT / 'models/batch-06/exp06/outputs'

## Validasi paket, fitur sensor dan metrik

In [ ]:
import hashlib
import json
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import RobustScaler

HRV = ["rmssd_ms", "sdnn_ms"]
EYE = ["blink_rate_per_minute", "ear_left_mean", "ear_left_std", "ear_right_mean",
       "ear_right_std", "iris_size_estimate_mean", "iris_size_estimate_std", "iris_size_delta_mean"]
CORE = HRV + EYE
ID = ["participant_id", "episode_id"]
SEED = 42


def read_bundle(root, case):
    proxy = case == "v3"
    package = root / ("v3_behavioral_proxy" if proxy else "v4_self_reported_rating")
    target = "cognitive_load_label" if proxy else "difficulty_rating"
    labels = ["Low", "Medium", "High"] if proxy else list(range(1, 11))
    window_dir = Path("window_15s_step_5s")
    hashes = {}

    def read(relative, columns):
        path = package / relative
        hashes[str(Path(relative).as_posix())] = hashlib.sha256(path.read_bytes()).hexdigest()
        return pd.read_csv(path, usecols=columns)

    def attach(left, right, keys, validation):
        merged = left.merge(right, on=keys, how="left", validate=validation, indicator=True)
        if not merged.pop("_merge").eq("both").all():
            raise ValueError(f"CSV pendamping tidak cocok pada {keys}.")
        return merged

    keys = ["window_id", "participant_id", "session_id", target]
    data = read(window_dir / "good_acceptable.csv", keys + CORE)
    assignments = read(window_dir / "window_assignments.csv",
                       keys + ["episode_id"] + (["split"] if proxy else []))
    for table in (data, assignments):
        if table.window_id.duplicated().any() or table[keys].isna().any().any():
            raise ValueError("Kunci window tidak lengkap atau window_id berulang.")
    data = attach(data, assignments, keys, "one_to_one")
    evidence = read("episode_labels.csv" if proxy else "episode_ratings.csv",
                    ID + ["session_id", target])
    if evidence.episode_id.isna().any() or evidence.episode_id.duplicated().any():
        raise ValueError("episode_id pada bukti harus unik dan lengkap.")
    data = attach(data, evidence, ID + ["session_id", target], "many_to_one")
    if set(data[target]) != set(labels) or not np.isfinite(data[CORE].to_numpy(float)).all():
        raise ValueError("Periksa kelas target dan kelengkapan sepuluh fitur core.")
    if proxy:
        splits = read("participant_splits.csv", ["participant_id", "split"])
        if splits.participant_id.duplicated().any() or set(splits.split) != {"Train", "validation", "Test"}:
            raise ValueError("Periksa pembagian peserta v3.")
        data = attach(data, splits, ["participant_id", "split"], "many_to_one")
        weights = read(window_dir / "train_sample_weights.csv", ["window_id", *ID, target, "sample_weight"])
        train = data.split.eq("Train")
        if set(weights.window_id) != set(data.loc[train, "window_id"]) or weights.window_id.duplicated().any():
            raise ValueError("Daftar bobot berbeda dari window Train.")
        checked = attach(data.loc[train], weights, ["window_id", *ID, target], "one_to_one")
        if not np.isfinite(checked.sample_weight).all() or not checked.sample_weight.gt(0).all():
            raise ValueError("Bobot Train harus positif dan finite.")
        data["sample_weight"] = data.window_id.map(checked.set_index("window_id").sample_weight)
    return data.rename(columns={target: "actual"}), labels, hashes

def scores(table, labels):
    actual, predicted = table.actual, table.predicted
    row = dict(n=len(table), participants=table.participant_id.nunique(),
               accuracy=accuracy_score(actual, predicted),
               macro_f1=f1_score(actual, predicted, labels=labels, average="macro", zero_division=0))
    if len(labels) == 10:
        distance = (actual - predicted).abs()
        row.update(mae=distance.mean(), within_one=distance.le(1).mean())
    return row

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")

## Fitur relatif terhadap riwayat sesi dan pembandingan berpasangan

In [ ]:
"""Paired single-modality controls using already recorded, labeled training episodes."""

HR_FAILURES = {"invalid_or_missing:rmssd_ms", "invalid_or_missing:sdnn_ms", "hrv_unusable",
               "hrv_coverage", "hrv_rr_pairs", "hrv_data_age_seconds", "rr_local_median_jump"}


def recoverable(root, case, source, out):
    package = root / ("v3_behavioral_proxy" if case == "v3" else "v4_self_reported_rating")
    target = "cognitive_load_label" if case == "v3" else "difficulty_rating"
    wd = package / "window_15s_step_5s"
    assignment = pd.read_csv(wd / "window_assignments.csv")
    accepted = pd.read_csv(wd / "good_acceptable.csv", usecols=["window_id"])
    covered = assignment.loc[assignment.window_id.isin(accepted.window_id), "episode_id"]
    candidates = assignment[assignment.quality_tier.eq("poor") & ~assignment.episode_id.isin(covered)]
    keys = ["window_id", "participant_id", "session_id", "quality_tier"]
    matched = candidates.merge(source, on=keys, validate="one_to_one")
    if len(matched) != len(candidates):
        raise ValueError("Sumber v2 tidak mencakup kandidat pemulihan.")
    evidence_file = "episode_labels.csv" if case == "v3" else "episode_ratings.csv"
    ids = ["episode_id", "participant_id", "session_id", target]
    ev = pd.read_csv(package / evidence_file, usecols=ids)
    if len(matched.merge(ev, on=ids, validate="many_to_one")) != len(matched):
        raise ValueError("Label episode kandidat berbeda dari bukti.")
    if case == "v3":
        splits = pd.read_csv(package / "participant_splits.csv")
        if len(matched.merge(splits, on=["participant_id", "split"], validate="many_to_one")) != len(matched):
            raise ValueError("Split peserta pada pemulihan berbeda dari split resmi.")
        matched = matched[matched.split.eq("Train")]
    flags = matched.acceptable_failed_criteria.fillna("").map(lambda s: set(s.split(";")))
    masks = {"eye": (EYE, flags.map(lambda f: f.issubset(HR_FAILURES))),
             "hrv": (HRV, flags.map(lambda f: f == {"min_iris_available_ratio"}))}
    extras = {}
    audit = []
    for name, (features, mask) in masks.items():
        usable = matched.loc[mask & np.isfinite(matched[features]).all(axis=1)].copy()
        usable = usable.rename(columns={target: "actual"})
        episode = usable.groupby("episode_id")
        extras[name] = episode[features].median().join(episode[["participant_id", "actual"]].first())
        audit.append(usable[["window_id", "episode_id", "participant_id", "actual", "acceptable_failed_criteria"]]
                     .assign(modality=name))
    pd.concat(audit).to_csv(out / f"{case}_recovered_windows.csv", index=False)
    return extras


def recovery_fold(case, fold, meta, x, extras, fit, held):
    predictions = []
    for name, features in [("eye", EYE), ("hrv", HRV)]:
        original = x.loc[fit, features].join(meta[["participant_id", "actual"]])
        additional = extras[name]
        # V3 additions already obey Train. For LOPO exclude the held person from all sources.
        additional = additional[~additional.participant_id.isin(meta.loc[held, "participant_id"])]
        for variant, train in [("original", original), ("expanded", pd.concat([original, additional]))]:
            if train.index.has_duplicates or train.participant_id.isin(meta.loc[held, "participant_id"]).any():
                raise ValueError("Episode duplikat atau peserta evaluasi muncul dalam fitting.")
            if case == "v3":
                participants = train.groupby("actual").participant_id.transform("nunique")
                episodes = train.groupby(["actual", "participant_id"]).actual.transform("size")
                weights = 1 / (participants * episodes)
                weights = weights / weights.mean()
                if variant == "original":
                    published = meta.loc[train.index, "weight"]
                    if not np.allclose(weights, published / published.mean()):
                        raise ValueError("Bobot kontrol tidak cocok dengan bobot episode resmi.")
            else:
                weights = np.ones(len(train))
            scaler = StandardScaler().fit(train[features], sample_weight=weights)
            model = SVC(C=1, kernel="rbf", gamma="scale")
            model.fit(scaler.transform(train[features]), train.actual, sample_weight=weights)
            result = meta.loc[held, ["participant_id", "actual", "history_windows"]].reset_index()
            result["predicted"] = model.predict(scaler.transform(x.loc[held, features]))
            predictions.append(result.assign(model=f"{name}_{variant}", fold=fold, fitting_episodes=len(train)))
    return predictions


"""Notebook cells. read_bundle, CORE, scores and save_json come from the shared relabeling pipeline."""

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

HEAD = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
SENSORS = CORE + HEAD


def history_streams(path):
    keys = ["participant_id", "session_id"]
    bank = pd.read_csv(path, usecols=keys + ["window_start", "window_end"] + SENSORS)
    for key in ["window_start", "window_end"]:
        bank[key] = pd.to_datetime(bank[key], utc=True, format="mixed", errors="raise")
    if bank[keys + ["window_start", "window_end"]].isna().any().any() or not bank.window_end.gt(bank.window_start).all():
        raise ValueError("Interval sensor historis tidak valid.")
    bank = bank[np.isfinite(bank[SENSORS]).all(axis=1)]
    return dict(tuple(bank.sort_values("window_end", ascending=False).groupby(keys)))


def prior_reference(stream, start):
    # Newest first. Work only with disjoint windows that finished before this task.
    chosen = []
    limit = start
    if stream is not None:
        for index, left, right in stream[["window_start", "window_end"]].itertuples():
            if right <= limit:
                chosen.append(index)
                limit = left
                if len(chosen) == 12:
                    break
    if len(chosen) < 3:
        return pd.Series(np.nan, index=SENSORS), len(chosen), None
    return stream.loc[chosen, SENSORS].median(), len(chosen), stream.at[chosen[0], "window_end"]


def cohort(root, case, streams):
    windows, labels, hashes = read_bundle(root, case)
    if case == "v3":
        windows = windows[windows.split.isin(["Train", "validation"])]
    package = root / ("v3_behavioral_proxy" if case == "v3" else "v4_self_reported_rating")
    source = package / "window_15s_step_5s/good_acceptable.csv"
    head = pd.read_csv(source, usecols=["window_id"] + HEAD)
    windows = windows.merge(head, on="window_id", how="left", validate="one_to_one")
    if not np.isfinite(windows[SENSORS]).all().all():
        raise ValueError("Perbandingan berpasangan memerlukan head pose dan core yang lengkap.")
    groups = windows.groupby("episode_id")
    x = groups[SENSORS].median()
    columns = ["participant_id", "session_id", "actual"] + (["split"] if case == "v3" else [])
    if groups[columns].nunique().ne(1).any().any():
        raise ValueError("Metadata episode tidak konsisten.")
    meta = groups[columns].first()
    if case == "v3":
        meta["weight"] = groups.sample_weight.sum(min_count=1)
    evidence = package / ("episode_labels.csv" if case == "v3" else "episode_ratings.csv")
    start_ms = pd.read_csv(evidence, usecols=["episode_id", "start_ms"]).set_index("episode_id").start_ms
    meta["start"] = pd.to_datetime(start_ms.loc[meta.index], unit="ms", utc=True, errors="raise")
    if meta.start.isna().any():
        raise ValueError("Awal episode tidak lengkap.")
    references = []
    for row in meta.itertuples():
        stream = streams.get((row.participant_id, row.session_id))
        ref, count, end = prior_reference(stream, row.start)
        references.append(ref)
        meta.at[row.Index, "history_windows"] = count
        meta.at[row.Index, "reference_end"] = end.isoformat() if end is not None else ""
    reference = pd.DataFrame(references, index=meta.index)
    return meta, x, reference, labels, hashes


def predict_fold(case, fold, meta, x, reference, labels, fit, held):
    train, test = meta.loc[fit], meta.loc[held]
    if set(train.participant_id) & set(test.participant_id):
        raise ValueError("Peserta fitting dan evaluasi tumpang tindih.")
    target = train.actual
    weights = train.weight / train.weight.mean() if case == "v3" else np.ones(len(train))
    # Cold start uses the same fold's fitting median, never other held episodes.
    centered = x - reference.fillna(x.loc[fit].median())
    predictions = []

    def record(name, values):
        frame = test[["participant_id", "actual", "history_windows"]].reset_index()
        predictions.append(frame.assign(model=name, predicted=values, fold=fold))

    record("dummy_mode", target.value_counts().reindex(labels, fill_value=0).idxmax())
    if case == "v4":
        record("dummy_median", target.quantile(0.5, interpolation="lower"))
    for name, view in [("raw_core", x[CORE]), ("relative_core", centered[CORE]),
                       ("raw_core_head", x), ("relative_core_head", centered)]:
        scaler = StandardScaler().fit(view.loc[fit], sample_weight=weights)
        x_fit, x_held = scaler.transform(view.loc[fit]), scaler.transform(view.loc[held])
        nominal = SVC(C=1, kernel="rbf", gamma="scale")
        nominal.fit(x_fit, target, sample_weight=weights)
        record("svc_" + name, nominal.predict(x_held))
        if case == "v4":
            ordinal = SVR(C=2, epsilon=0.5, kernel="rbf", gamma="scale")
            ordinal.fit(x_fit, target)
            record("svr_" + name, np.floor(ordinal.predict(x_held).clip(1, 10) + 0.5).astype(int))
    return predictions


def run_history(root, output_root):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out = output_root / f"history_{stamp}_{uuid4().hex[:8]}"
    bank_path = root / "v2_session_phase/window_15s_step_5s/good_acceptable.csv"
    streams = history_streams(bank_path)
    all_path = bank_path.with_name("all_data.csv")
    source = pd.read_csv(all_path, usecols=["window_id", "participant_id", "session_id", "quality_tier",
                                           "acceptable_failed_criteria"] + CORE)
    out.mkdir(parents=True, exist_ok=False)
    inputs = {"v2_history_sha256": hashlib.sha256(bank_path.read_bytes()).hexdigest(),
              "v2_all_sha256": hashlib.sha256(all_path.read_bytes()).hexdigest()}
    metrics, per_person = [], []
    for case in ["v3", "v4"]:
        meta, x, ref, labels, inputs[case] = cohort(root, case, streams)
        extras = recoverable(root, case, source, out)
        if case == "v3":
            folds = [("validation", meta.split.eq("Train"), meta.split.eq("validation"))]
        else:
            folds = [(f"LOPO_{i}", meta.participant_id.ne(p), meta.participant_id.eq(p))
                     for i, p in enumerate(sorted(meta.participant_id.unique()), 1)]
        predictions = []
        for fold, fit, held in folds:
            predictions.extend(predict_fold(case, fold, meta, x, ref, labels, fit, held))
            predictions.extend(recovery_fold(case, fold, meta, x, extras, fit, held))
        pred = pd.concat(predictions, ignore_index=True)
        expected = meta.index[meta.split.eq("validation")] if case == "v3" else meta.index
        for name, group in pred.groupby("model"):
            if group.episode_id.duplicated().any() or set(group.episode_id) != set(expected):
                raise ValueError("Cakupan prediksi salah.")
            info = {"case": case, "model": name}
            metrics.append({**info, **scores(group, labels), "correct": int(group.actual.eq(group.predicted).sum())})
            for person, rows in group.groupby("participant_id"):
                per_person.append({**info, "participant_id": person, **scores(rows, labels)})
        meta.to_csv(out / f"{case}_references.csv")
        pred.to_csv(out / f"{case}_predictions.csv", index=False)
    summary = pd.DataFrame(metrics)
    summary.to_csv(out / "metrics.csv", index=False)
    pd.DataFrame(per_person).to_csv(out / "participant_metrics.csv", index=False)
    save_json(out / "manifest.json", {"inputs": inputs, "features": SENSORS, "sklearn": sklearn.__version__,
              "pandas": pd.__version__, "numpy": np.__version__, "primary_exact": "svc_relative_core_head",
              "primary_distance": "svr_relative_core_head", "max_reference_windows": 12, "min_reference_windows": 3,
              "svc_C": 1, "svr_C": 2, "svr_epsilon": 0.5, "kernel": "rbf", "gamma": "scale"})
    print(summary.to_string(index=False))
    archive = shutil.make_archive(str(out), "zip", root_dir=out.parent, base_dir=out.name)
    print("ZIP:", archive)
    return Path(archive)


## Jalankan kedua kasus dan simpan ZIP

In [ ]:
archive = run_history(DATASET_ROOT, OUTPUT_ROOT)